# RAG + LLM Source Mapping Pipeline

This pipeline combines:
1. **RAG Source Finding**: Retrieve top 10 candidate sources using embedding similarity
2. **LLM Source Mapping**: Generate source properties for each candidate using `SourcePropertyFind_mapWithDescription`
3. **LLM Selection**: Select best 3 sources based on generated mappings


## Setup and Imports


In [1]:
import sys
import os
import pandas as pd
import numpy as np
import json
import re
import ast
from dotenv import load_dotenv
import dspy
from tqdm.auto import tqdm

# Load environment variables
load_dotenv()

# Add paths for imports
sys.path.append('../../stage1_analysis/source_finding')
sys.path.append('../../stage1_analysis/mapping_generation')

from rag_source_finder import RAGSourceFinder
from easy_llm_importer import create_client, DSPyAdapter, LLMClient

# Data path
DATA_PATH = "../../data/SCAR_cleaned_manually.csv"

print("Imports loaded successfully!")


Imports loaded successfully!


c:\Users\maria\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Configure LLM client
client = create_client()
llm_client = LLMClient()

# Configure DSPy with Llama 405B
MODEL_NAME = "llama-3.1-405b-instruct"
adapter = DSPyAdapter(client, model_name=MODEL_NAME)
lm = adapter.get_dspy_lm()
dspy.settings.configure(lm=lm)

print(f"DSPy configured with {MODEL_NAME}")


DSPy configured with llama-3.1-405b-instruct


## DSPy Signature for Source Property Mapping


In [3]:
# Source Property Mapping Signature (from Mapping_analysis_v3.ipynb)
class SourcePropertyFind_mapWithDescription(dspy.Signature):
    """Given an unfamiliar concept with its properties and descriptions, find corresponding properties from a familiar concept."""
    
    unfamiliar_concept: str = dspy.InputField(desc="The unfamiliar concept for student to learn")
    description_of_unfamiliar_concept: str = dspy.InputField(desc="Detailed description or context about the unfamiliar concept")
    properties_of_unfamiliar_concept: list[str] = dspy.InputField(desc="List of key properties that characterize the unfamiliar concept. Each property is 1-2 words maximum.")
    familiar_concept: str = dspy.InputField(desc="The familiar concept used to create the analogy")
    description_of_familiar_concept: str = dspy.InputField(desc="Detailed description or context about the familiar concept")
    mapped_source_properties: dict[str, str] = dspy.OutputField(desc="Dictionary mapping each unfamiliar concept property to corresponding familiar concept property (1-2 words each)")

print("DSPy signature defined!")


DSPy signature defined!


## Initialize RAG Source Finder


In [4]:
# Initialize RAG source finder
finder = RAGSourceFinder(embedding_mode="name_properties")
finder.load_corpus_from_csv(DATA_PATH)
finder.embed_corpus()

print(f"RAG finder ready with {len(finder.corpus_df)} sources")


Loaded 333 unique sources from corpus (mode: name_properties)
Generating embeddings for corpus...
Generated 333 embeddings
RAG finder ready with 333 sources


In [5]:
# Load SCAR dataset
df_scar = pd.read_csv(DATA_PATH)

# Build source description lookup
source_desc_lookup = {}
for _, row in df_scar.iterrows():
    source_name = row['system_b']
    if source_name not in source_desc_lookup:
        source_desc_lookup[source_name] = row['system_b_background'] if pd.notna(row['system_b_background']) else ""

# Helper to extract target properties from mappings_parsed
def get_target_properties(row):
    if pd.notna(row.get('mappings_parsed')):
        try:
            mappings = ast.literal_eval(row['mappings_parsed'])
            props = [pair[0] for pair in mappings if len(pair) > 0]
            return props
        except:
            pass
    return []

print(f"Loaded {len(df_scar)} examples from SCAR dataset")
print(f"Source descriptions available: {len(source_desc_lookup)}")


Loaded 400 examples from SCAR dataset
Source descriptions available: 333


## Pipeline Functions


In [6]:
import time
import traceback

# =============================================================================
# CONFIGURATION
# =============================================================================
ERROR_MARKER = "ERROR"
DEBUG_VERBOSE = True  # Set to True for verbose debug output

# =============================================================================
# DSPy SIGNATURE FOR RERANKER (Selects by NAME, not index)
# =============================================================================
RERANKER_INSTRUCTIONS = """You are selecting the best analogous source concepts for a scientific analogy.

Your task:
1. Analyze the target concept and its properties
2. Review each candidate source and its generated analogous properties
3. Select the 3 candidates whose properties BEST map to the target properties

Selection criteria:
- Strong structural/functional correspondence between source and target properties
- The source concept should be familiar and help explain the unfamiliar target
- Prefer sources with clear, well-mapped properties over vague ones

Return the EXACT names of your top 3 selected candidates."""


class SourceRerankerSignature(dspy.Signature):
    """Select the best source concepts for a scientific analogy based on property mappings."""
    
    target_concept: str = dspy.InputField(desc="The unfamiliar target concept to find analogies for")
    target_properties: str = dspy.InputField(desc="Comma-separated properties of the target concept")
    candidates_info: str = dspy.InputField(desc="List of candidate sources with their generated analogous properties")
    selected_sources: list[str] = dspy.OutputField(desc="List of exactly 3 selected candidate names (use exact names from the list)")


# =============================================================================
# CONCEPT GENERATION (with reasoning capture)
# =============================================================================
def generate_source_mappings(target_name, target_desc, target_properties, source_name, source_desc, max_retries=3):
    """
    Use LLM to generate source properties that map to target properties.
    
    Returns:
        tuple: (mappings_dict, reasoning_str) on success
               (ERROR_MARKER, error_message) on failure
    """
    last_error = None
    
    for attempt in range(max_retries):
        try:
            predictor = dspy.ChainOfThought(SourcePropertyFind_mapWithDescription)
            result = predictor(
                unfamiliar_concept=target_name,
                description_of_unfamiliar_concept=target_desc,
                properties_of_unfamiliar_concept=target_properties,
                familiar_concept=source_name,
                description_of_familiar_concept=source_desc
            )
            
            # Debug output
            if DEBUG_VERBOSE:
                print(f"\n  [MAPPING] {source_name}")
                print(f"    Input: target='{target_name}', props={target_properties}")
                print(f"    Output: {result.mapped_source_properties}")
                print(f"    Reasoning: {result.reasoning[:150]}..." if len(result.reasoning) > 150 else f"    Reasoning: {result.reasoning}")
            
            return (result.mapped_source_properties, result.reasoning)
            
        except Exception as e:
            last_error = str(e)
            is_retryable = any(x in last_error.lower() for x in [
                'truncated', 'failed to parse', 'rate limit', 'timeout', 'connection'
            ])
            
            if DEBUG_VERBOSE:
                print(f"\n  [MAPPING ERROR] Attempt {attempt + 1}/{max_retries} for '{source_name}'")
                print(f"    Error: {last_error[:200]}...")
            
            if is_retryable and attempt < max_retries - 1:
                time.sleep((attempt + 1) * 2)
            else:
                return (ERROR_MARKER, last_error)
    
    return (ERROR_MARKER, last_error)


# =============================================================================
# RERANKER (DSPy-based, selects by name)
# =============================================================================
def llm_select_best_sources(target_name, target_properties, candidates_with_mappings, top_k=3, max_retries=3):
    """
    Use DSPy to select best sources from candidates (by NAME, not index).
    
    Args:
        candidates_with_mappings: list of (source_name, (mappings_dict, reasoning) or (ERROR_MARKER, error))
    
    Returns:
        tuple: (selected_names_list, reasoning_str) on success
               (ERROR_MARKER, error_message) on failure
    """
    # Build candidate names list and formatted text
    candidate_names = [name for name, _ in candidates_with_mappings]
    
    candidates_text = "\n".join([
        f"- {name}: {', '.join(data[0].values()) if isinstance(data, tuple) and isinstance(data[0], dict) else 'No properties generated'}"
        for name, data in candidates_with_mappings
    ])
    
    # Debug: print reranker inputs
    if DEBUG_VERBOSE:
        print(f"\n{'='*60}")
        print(f"[RERANKER INPUT]")
        print(f"  Target: {target_name}")
        print(f"  Target Properties: {', '.join(target_properties)}")
        print(f"  Candidates:\n{candidates_text}")
        print(f"{'='*60}")
    
    last_error = None
    
    for attempt in range(max_retries):
        try:
            # Create signature with instructions
            signature = dspy.Signature(
                "target_concept: str, target_properties: str, candidates_info: str -> selected_sources: list[str]",
                instructions=RERANKER_INSTRUCTIONS
            )
            
            reranker = dspy.ChainOfThought(signature)
            result = reranker(
                target_concept=target_name,
                target_properties=", ".join(target_properties),
                candidates_info=candidates_text
            )
            
            # Validate selected names exist in candidates (fuzzy match)
            selected_raw = result.selected_sources
            selected_valid = []
            
            for sel_name in selected_raw:
                # Try exact match first
                if sel_name in candidate_names:
                    selected_valid.append(sel_name)
                else:
                    # Try case-insensitive match
                    for cand_name in candidate_names:
                        if sel_name.lower() == cand_name.lower():
                            selected_valid.append(cand_name)
                            break
                        # Partial match (selection contains candidate or vice versa)
                        elif sel_name.lower() in cand_name.lower() or cand_name.lower() in sel_name.lower():
                            selected_valid.append(cand_name)
                            break
            
            # Debug output
            if DEBUG_VERBOSE:
                print(f"\n[RERANKER OUTPUT]")
                print(f"  Selected (raw): {selected_raw}")
                print(f"  Selected (validated): {selected_valid}")
                print(f"  Reasoning: {result.reasoning[:200]}..." if len(result.reasoning) > 200 else f"  Reasoning: {result.reasoning}")
            
            if selected_valid:
                return (selected_valid[:top_k], result.reasoning)
            else:
                if attempt < max_retries - 1:
                    print(f"  [RERANKER] No valid selections found, retrying...")
                    time.sleep((attempt + 1) * 2)
                    
        except Exception as e:
            last_error = str(e)
            if DEBUG_VERBOSE:
                print(f"\n  [RERANKER ERROR] Attempt {attempt + 1}/{max_retries}")
                print(f"    Error: {last_error[:300]}...")
            
            if attempt < max_retries - 1:
                time.sleep((attempt + 1) * 2)
    
    return (ERROR_MARKER, last_error or "No valid selections after all retries")


# =============================================================================
# MAIN PIPELINE
# =============================================================================
def run_pipeline(row):
    """
    Run full pipeline on a single row:
    1. RAG retrieval (top 10)
    2. Generate source mappings for each candidate (with reasoning)
    3. LLM reranker selects best 3 (with reasoning)
    
    Returns dict with status and all reasoning captured.
    """
    target_name = row['system_a']
    target_desc = row['system_a_background'] if pd.notna(row['system_a_background']) else ""
    target_properties = get_target_properties(row)
    gold_source = row['system_b']
    
    if DEBUG_VERBOSE:
        print(f"\n{'#'*70}")
        print(f"# Processing: {target_name}")
        print(f"# Gold source: {gold_source}")
        print(f"{'#'*70}")
    
    try:
        # Step 1: RAG retrieval
        candidates = finder.find_source(
            target_name=target_name,
            target_background=target_desc,
            target_properties=", ".join(target_properties),
            top_k=10
        )
        
        if DEBUG_VERBOSE:
            print(f"\n[RAG] Retrieved {len(candidates)} candidates:")
            for i, c in enumerate(candidates):
                print(f"  {i+1}. {c.name} (score: {c.similarity_score:.3f})")
        
        # Step 2: Generate source mappings for each candidate
        candidates_with_mappings = []  # List of (name, (mappings, reasoning))
        mapping_reasonings = []
        mapping_errors = 0
        
        for candidate in candidates:
            source_desc = source_desc_lookup.get(candidate.name, "")
            result = generate_source_mappings(
                target_name, target_desc, target_properties,
                candidate.name, source_desc
            )
            
            mappings, reasoning = result
            if mappings == ERROR_MARKER:
                mapping_errors += 1
                mapping_reasonings.append(f"ERROR: {reasoning}")
            else:
                mapping_reasonings.append(reasoning)
            
            candidates_with_mappings.append((candidate.name, result))
        
        # Step 3: LLM reranker selects best 3
        selected_result = llm_select_best_sources(
            target_name, target_properties,
            candidates_with_mappings, top_k=3
        )
        
        selected_names, reranker_reasoning = selected_result
        
        # Check if selection failed
        if selected_names == ERROR_MARKER:
            return {
                'id': row['id'],
                'target': target_name,
                'target_properties': target_properties,
                'gold_source': gold_source,
                'top_10_sources': [c.name for c in candidates],
                'top_10_scores': [c.similarity_score for c in candidates],
                'candidates_with_mappings': str([(n, m[0]) for n, m in candidates_with_mappings]),
                'mapping_reasonings': mapping_reasonings,
                'selected_sources': ERROR_MARKER,
                'selected_mappings': ERROR_MARKER,
                'reranker_reasoning': reranker_reasoning,
                'gold_rank': ERROR_MARKER,
                'status': 'selection_error',
                'mapping_errors': mapping_errors
            }
        
        # Get selected mappings
        selected_mappings = []
        for sel_name in selected_names:
            for cand_name, (mappings, _) in candidates_with_mappings:
                if cand_name == sel_name:
                    selected_mappings.append(mappings if mappings != ERROR_MARKER else {})
                    break
        
        # Find gold rank in selected
        gold_rank = -1
        for i, name in enumerate(selected_names):
            if gold_source.lower() in name.lower() or name.lower() in gold_source.lower():
                gold_rank = i + 1
                break
        
        status = 'partial_mapping_error' if mapping_errors > 0 else 'success'
        
        return {
            'id': row['id'],
            'target': target_name,
            'target_properties': target_properties,
            'gold_source': gold_source,
            'top_10_sources': [c.name for c in candidates],
            'top_10_scores': [c.similarity_score for c in candidates],
            'candidates_with_mappings': str([(n, m[0]) for n, m in candidates_with_mappings]),
            'mapping_reasonings': mapping_reasonings,
            'selected_sources': selected_names,
            'selected_mappings': selected_mappings,
            'reranker_reasoning': reranker_reasoning,
            'gold_rank': gold_rank,
            'status': status,
            'mapping_errors': mapping_errors
        }
        
    except Exception as e:
        error_msg = str(e)
        print(f"\n{'='*60}")
        print(f"[PIPELINE ERROR] Unexpected error for '{target_name}'")
        print(f"Error: {error_msg}")
        traceback.print_exc()
        print(f"{'='*60}\n")
        
        return {
            'id': row['id'],
            'target': target_name,
            'target_properties': target_properties,
            'gold_source': gold_source,
            'top_10_sources': ERROR_MARKER,
            'top_10_scores': ERROR_MARKER,
            'candidates_with_mappings': ERROR_MARKER,
            'mapping_reasonings': ERROR_MARKER,
            'selected_sources': ERROR_MARKER,
            'selected_mappings': ERROR_MARKER,
            'reranker_reasoning': ERROR_MARKER,
            'gold_rank': ERROR_MARKER,
            'status': 'pipeline_error',
            'mapping_errors': -1,
            'error_message': error_msg
        }

print("Pipeline functions defined!")


Pipeline functions defined!


## Run Pipeline on SCAR Dataset


In [7]:
# Run pipeline on full dataset
results = []

print(f"Running pipeline on {len(df_scar)} examples...")
for idx, row in tqdm(df_scar.iterrows(), total=len(df_scar), desc="Processing"):
    result = run_pipeline(row)
    results.append(result)
    
    # Progress update every 50
    if (idx + 1) % 50 == 0:
        # Only count successful results for hit rate
        successful = [r for r in results if r['status'] == 'success' or r['status'] == 'partial_mapping_error']
        errors = [r for r in results if r['status'] == 'selection_error']
        if successful:
            found_so_far = sum(1 for r in successful if isinstance(r['gold_rank'], int) and r['gold_rank'] > 0)
            hit_rate = 100 * found_so_far / len(successful)
        else:
            hit_rate = 0
        print(f"Progress: {idx+1}/{len(df_scar)} - Hit rate: {hit_rate:.1f}% (errors: {len(errors)})")

# Final summary
successful_results = [r for r in results if r['status'] in ['success', 'partial_mapping_error']]
error_results = [r for r in results if r['status'] == 'selection_error']
print(f"\nPipeline complete!")
print(f"  Total processed: {len(results)}")
print(f"  Successful: {len(successful_results)}")
print(f"  Errors: {len(error_results)}")


Running pipeline on 400 examples...


Processing:   0%|          | 0/400 [00:00<?, ?it/s]


######################################################################
# Processing: biological clock
# Gold source: clock
######################################################################

[RAG] Retrieved 10 candidates:
  1. clock (score: 0.487)
  2. Biological Evolution (score: 0.473)
  3. Time Machine (score: 0.431)
  4. Evolution (score: 0.408)
  5. Operating System (score: 0.391)
  6. day and night cycle (score: 0.387)
  7. bacterial mutation (score: 0.382)
  8. electronic scale automatically adjusts (score: 0.366)
  9. ecosystem (score: 0.359)
  10. Human Body (score: 0.359)

  [MAPPING] clock
    Input: target='biological clock', props=['changes', 'state', 'adjust']
    Output: {'changes': 'movement', 'state': 'time', 'adjust': 'setting'}
    Reasoning: The biological clock and a traditional clock share similarities in their ability to govern and regulate processes. A clock is used to measure time, wh...

  [MAPPING] Biological Evolution
    Input: target='biological clock

Processing:   0%|          | 1/400 [00:20<2:14:34, 20.24s/it]


[RERANKER OUTPUT]
  Selected (raw): ['clock', 'electronic scale automatically adjusts', 'Human Body']
  Selected (validated): ['clock', 'electronic scale automatically adjusts', 'Human Body']
  Reasoning: The target concept "biological clock" has properties related to changes, state, and adjustment. The best analogous source concepts should have properties that map well to these, showing a strong struc...

######################################################################
# Processing: Biosphere
# Gold source: Library
######################################################################

[RAG] Retrieved 10 candidates:
  1. Ecosystem (score: 0.742)
  2. ecosystem (score: 0.601)
  3. Biological Evolution (score: 0.464)
  4. forest (score: 0.445)
  5. Evolution (score: 0.426)
  6. Desert (score: 0.415)
  7. life (score: 0.382)
  8. Greenhouse (score: 0.368)
  9. universe (score: 0.358)
  10. respiration (score: 0.357)

  [MAPPING] Ecosystem
    Input: target='Biosphere', props=['bio

...


[RERANKER OUTPUT]
  Selected (raw): ['Sun Protection', 'Reactor', 'Firewall']
  Selected (validated): ['Sun Protection', 'Reactor', 'Firewall']
  Reasoning: The target concept is Radiation Protection, with properties radiation, shield, and dose. To find the best analogous source concepts, we need to look for candidates that have strong structural and func...

######################################################################
# Processing: Proton Accelerator
# Gold source: The Hunt
######################################################################

[RAG] Retrieved 10 candidates:
  1. Reactor (score: 0.491)
  2. atom (score: 0.399)
  3. planet (score: 0.356)
  4. Competition (score: 0.353)
  5. Computer Processor (score: 0.340)
  6. Power Generation (score: 0.340)
  7. electromagnetic emission system (score: 0.325)
  8. clock (score: 0.324)
  9. universe (score: 0.323)
  10. engine (score: 0.321)

  [MAPPING] Reactor
    Input: target='Proton Accelerator', props=['proton', 'acce

Processing: 100%|█████████▉| 399/400 [12:06:12<01:48, 108.66s/it]


[RERANKER OUTPUT]
  Selected (raw): ['atom', 'engine', 'Power Generation']
  Selected (validated): ['atom', 'engine', 'Power Generation']
  Reasoning: To select the best analogous source concepts for the Proton Accelerator, we need to analyze the target properties: proton, accelerator, and target. We are looking for sources with strong structural/fu...

######################################################################
# Processing: Particle Accelerator
# Gold source: The Ring
######################################################################

[RAG] Retrieved 10 candidates:
  1. Reactor (score: 0.454)
  2. atom (score: 0.430)
  3. electromagnetic emission system (score: 0.398)
  4. camera (score: 0.360)
  5. engine (score: 0.353)
  6. Shooting Through Walls (score: 0.352)
  7. Computer Processor (score: 0.351)
  8. planet (score: 0.349)
  9. Firewall (score: 0.347)
  10. Power Generation (score: 0.340)

  [MAPPING] Reactor
    Input: target='Particle Accelerator', props=['part

Processing: 100%|██████████| 400/400 [12:08:37<00:00, 109.29s/it]


[RERANKER OUTPUT]
  Selected (raw): ['Reactor', 'electromagnetic emission system', 'engine']
  Selected (validated): ['Reactor', 'electromagnetic emission system', 'engine']
  Reasoning: To select the best analogous source concepts for the target concept "Particle Accelerator", we need to analyze the properties of the target concept and find the candidates whose properties have a stro...
Progress: 400/400 - Hit rate: 33.7% (errors: 11)

Pipeline complete!
  Total processed: 400
  Successful: 389
  Errors: 11


In [13]:
# Convert to DataFrame and save
results_df = pd.DataFrame(results)

# Save full results
results_df.to_csv('pipeline_results.csv', index=False)
print(f"Results saved to pipeline_results.csv")


Results saved to pipeline_results.csv


## Evaluation Metrics


In [9]:
# Compute metrics for this pipeline - EXCLUDING ERROR CASES
total = len(results_df)

# Separate successful and error cases
successful_df = results_df[results_df['status'].isin(['success', 'partial_mapping_error'])]
error_df = results_df[results_df['status'] == 'selection_error']

successful_count = len(successful_df)
error_count = len(error_df)

# Compute metrics only on successful cases
if successful_count > 0:
    # gold_rank is now guaranteed to be numeric for successful cases
    found = (successful_df['gold_rank'] > 0).sum()
    hit_rate = 100 * found / successful_count
    
    # Mean rank when found
    found_df = successful_df[successful_df['gold_rank'] > 0]
    mean_rank = found_df['gold_rank'].mean() if len(found_df) > 0 else 0
    
    # MRR (Mean Reciprocal Rank)
    mrr = found_df['gold_rank'].apply(lambda x: 1/x).mean() if len(found_df) > 0 else 0
else:
    found = 0
    hit_rate = 0
    mean_rank = 0
    mrr = 0

print("=" * 60)
print("RAG + LLM SOURCE MAPPING PIPELINE RESULTS")
print("=" * 60)
print(f"\nTotal examples: {total}")
print(f"  - Successful: {successful_count}")
print(f"  - Errors (excluded from metrics): {error_count}")
print(f"\n[Metrics computed on {successful_count} successful examples only]")
print(f"Gold source found in top 3: {found}/{successful_count} ({hit_rate:.1f}%)")
if mean_rank > 0:
    print(f"Mean gold rank (when found): {mean_rank:.2f}")
    print(f"MRR (when found): {mrr:.3f}")
print("=" * 60)

# Show error breakdown if any
if error_count > 0:
    print(f"\n⚠️  WARNING: {error_count} examples failed and were EXCLUDED from metrics!")
    print("   These are marked with 'ERROR' in the results CSV.")


RAG + LLM SOURCE MAPPING PIPELINE RESULTS

Total examples: 400
  - Successful: 389
  - Errors (excluded from metrics): 11

[Metrics computed on 389 successful examples only]
Gold source found in top 3: 131/389 (33.7%)
Mean gold rank (when found): 1.63
MRR (when found): 0.739

⚠️  WARNING: 11 examples failed and were EXCLUDED from metrics!
   These are marked with 'ERROR' in the results CSV.


In [14]:
# =============================================================================
# RETRY ERROR RECORDS
# =============================================================================
if error_count > 0:
    print(f"\n{'='*60}")
    print(f"RETRYING {error_count} ERROR RECORDS")
    print(f"{'='*60}")
    
    # Get error record IDs
    error_ids = error_df['id'].tolist()
    print(f"\nError record IDs: {error_ids}")
    
    # Show error details before retry
    print(f"\nError records details:")
    for idx, row in error_df.iterrows():
        print(f"  - ID {row['id']}: target='{row['target']}', gold='{row['gold_source']}'")
        if 'reranker_reasoning' in row:
            reason = str(row['reranker_reasoning'])[:100]
            print(f"    Error reason: {reason}...")
    
    # Get original rows from df_scar for retry
    retry_rows = df_scar[df_scar['id'].isin(error_ids)]
    print(f"\nFound {len(retry_rows)} rows to retry in original dataset")
    
    # Rerun pipeline on error records
    retry_results = []
    for idx, row in tqdm(retry_rows.iterrows(), total=len(retry_rows), desc="Retrying errors"):
        print(f"\n>>> Retrying ID {row['id']}: {row['system_a']}")
        result = run_pipeline(row)
        retry_results.append(result)
        
        # Show retry status
        if result['status'] in ['success', 'partial_mapping_error']:
            print(f"    ✅ SUCCESS! Gold rank: {result['gold_rank']}")
        else:
            print(f"    ❌ Still failed: {result['status']}")
    
    # Count retry outcomes
    retry_success = sum(1 for r in retry_results if r['status'] in ['success', 'partial_mapping_error'])
    retry_still_error = sum(1 for r in retry_results if r['status'] == 'selection_error')
    
    print(f"\n{'='*60}")
    print(f"RETRY RESULTS SUMMARY")
    print(f"{'='*60}")
    print(f"Total retried: {len(retry_results)}")
    print(f"  - Now successful: {retry_success}")
    print(f"  - Still errored: {retry_still_error}")
    
    # Update results_df with retry results
    if retry_results:
        retry_df = pd.DataFrame(retry_results)
        
        # Remove old error records from results_df
        results_df_updated = results_df[~results_df['id'].isin(error_ids)].copy()
        
        # Append retry results
        results_df_updated = pd.concat([results_df_updated, retry_df], ignore_index=True)
        
        # Sort by ID
        results_df_updated = results_df_updated.sort_values('id').reset_index(drop=True)
        
        # Update the global results_df
        results_df = results_df_updated
        
        # Save updated results
        results_df.to_csv('pipeline_results.csv', index=False)
        print(f"\n✅ Updated results saved to pipeline_results.csv")
        
        # Recalculate metrics
        successful_df = results_df[results_df['status'].isin(['success', 'partial_mapping_error'])]
        error_df = results_df[results_df['status'] == 'selection_error']
        
        successful_count = len(successful_df)
        error_count = len(error_df)
        
        if successful_count > 0:
            found = (successful_df['gold_rank'] > 0).sum()
            hit_rate = 100 * found / successful_count
            found_df = successful_df[successful_df['gold_rank'] > 0]
            mean_rank = found_df['gold_rank'].mean() if len(found_df) > 0 else 0
            mrr = found_df['gold_rank'].apply(lambda x: 1/x).mean() if len(found_df) > 0 else 0
        
        print(f"\n{'='*60}")
        print(f"UPDATED METRICS (After Retry)")
        print(f"{'='*60}")
        print(f"Total examples: {len(results_df)}")
        print(f"  - Successful: {successful_count}")
        print(f"  - Errors (excluded): {error_count}")
        print(f"\nGold source found in top 3: {found}/{successful_count} ({hit_rate:.1f}%)")
        if mean_rank > 0:
            print(f"Mean gold rank (when found): {mean_rank:.2f}")
            print(f"MRR (when found): {mrr:.3f}")
        print(f"{'='*60}")
else:
    print("\n✅ No error records to retry!")



RETRYING 11 ERROR RECORDS

Error record IDs: [178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188]

Error records details:
  - ID 178: target='Atmosphere', gold='Greenhouse'
    Error reason: litellm.APIError: APIError: OpenAIException - This request requires more credits, or fewer max_token...
  - ID 179: target='Glacier', gold='Ice Cream'
    Error reason: litellm.APIError: APIError: OpenAIException - This request requires more credits, or fewer max_token...
  - ID 180: target='Tide', gold='lift'
    Error reason: litellm.APIError: APIError: OpenAIException - This request requires more credits, or fewer max_token...
  - ID 181: target='Active faults', gold='the skeletal system'
    Error reason: litellm.APIError: APIError: OpenAIException - This request requires more credits, or fewer max_token...
  - ID 182: target='Active Fault', gold='Spring'
    Error reason: litellm.APIError: APIError: OpenAIException - This request requires more credits, or fewer max_token...
  - ID 183: tar

Retrying errors:   0%|          | 0/11 [00:00<?, ?it/s]


>>> Retrying ID 178: Atmosphere

######################################################################
# Processing: Atmosphere
# Gold source: Greenhouse
######################################################################

[RAG] Retrieved 10 candidates:
  1. Greenhouse (score: 0.561)
  2. planet (score: 0.548)
  3. universe (score: 0.507)
  4. heat (score: 0.417)
  5. day and night cycle (score: 0.393)
  6. Sun Protection (score: 0.387)
  7. Plants (score: 0.383)
  8. The Real World (score: 0.357)
  9. Dust Storm (score: 0.353)
  10. Friction (score: 0.347)

  [MAPPING] Greenhouse
    Input: target='Atmosphere', props=['Earth', 'atmosphere', 'sun rays']
    Output: {'Earth': 'Greenhouse', 'atmosphere': 'cover', 'sun rays': 'sunlight'}
    Reasoning: To create an analogy between the atmosphere and a greenhouse, we need to identify the key properties of the atmosphere and map them to corresponding p...

  [MAPPING] planet
    Input: target='Atmosphere', props=['Earth', 'atmosphere',

Retrying errors:   9%|▉         | 1/11 [01:22<13:45, 82.58s/it]


[RERANKER OUTPUT]
  Selected (raw): ['Greenhouse', 'Plants', 'Sun Protection']
  Selected (validated): ['Greenhouse', 'Plants', 'Sun Protection']
  Reasoning: To select the best analogous source concepts for the target concept "Atmosphere", we need to analyze the properties of the target concept and the candidates. The target properties are "Earth", "atmosp...
    ✅ SUCCESS! Gold rank: 1

>>> Retrying ID 179: Glacier

######################################################################
# Processing: Glacier
# Gold source: Ice Cream
######################################################################

[RAG] Retrieved 10 candidates:
  1. Ice Cream (score: 0.422)
  2. Greenhouse (score: 0.336)
  3. The Great Wall (score: 0.333)
  4. Great Wall (score: 0.328)
  5. Walls (score: 0.323)
  6. forest (score: 0.293)
  7. gas molecules (score: 0.290)
  8. Greenhouses (score: 0.287)
  9. River (score: 0.286)
  10. universe (score: 0.283)

  [MAPPING] Ice Cream
    Input: target='Glacier', pr

...

Retrying errors:  91%|█████████ | 10/11 [15:26<01:29, 89.60s/it]


[RERANKER OUTPUT]
  Selected (raw): ['Prospecting', 'Archeology', 'Buildings']
  Selected (validated): ['Prospecting', 'Archeology', 'Buildings']
  Reasoning: To find the best analogous source concepts for "Deposit Formation", we need to analyze the target properties: crust, mineral, and ore deposit. These properties suggest a geological process involving t...
    ✅ SUCCESS! Gold rank: -1

>>> Retrying ID 188: Empire

######################################################################
# Processing: Empire
# Gold source: Tree
######################################################################

[RAG] Retrieved 10 candidates:
  1. Ancient Rome (score: 0.419)
  2. War (score: 0.406)
  3. Feudal Dynasties (score: 0.395)
  4. Ancient Western Asia (score: 0.395)
  5. Ancient Greece (score: 0.385)
  6. Great Wall (score: 0.373)
  7. Currency Loss (score: 0.372)
  8. Archeology (score: 0.362)
  9. life (score: 0.348)
  10. crime (score: 0.339)

  [MAPPING] Ancient Rome
    Input: target=

Retrying errors: 100%|██████████| 11/11 [16:40<00:00, 90.95s/it]


[RERANKER OUTPUT]
  Selected (raw): ['Ancient Rome', 'Ancient Western Asia', 'Ancient Greece']
  Selected (validated): ['Ancient Rome', 'Ancient Western Asia', 'Ancient Greece']
  Reasoning: The target concept "Empire" has properties "territory", "civilization", and "perish". To find the best analogous source concepts, we need to look for candidates that have similar properties. We can se...
    ✅ SUCCESS! Gold rank: -1

RETRY RESULTS SUMMARY
Total retried: 11
  - Now successful: 11
  - Still errored: 0



✅ Updated results saved to pipeline_results.csv

UPDATED METRICS (After Retry)
Total examples: 400
  - Successful: 400
  - Errors (excluded): 0

Gold source found in top 3: 136/400 (34.0%)
Mean gold rank (when found): 1.62
MRR (when found): 0.741


## Compare with Baselines


In [10]:
# Load baseline results for comparison
BASELINES_DIR = "../baselines/"

# Load all baseline CSVs
baseline1_df = pd.read_csv(f"{BASELINES_DIR}baseline1_embedding_top3.csv")
baseline2b_df = pd.read_csv(f"{BASELINES_DIR}baseline2b_name_only.csv")
baseline2c_df = pd.read_csv(f"{BASELINES_DIR}baseline2c_full.csv")
baseline2d_df = pd.read_csv(f"{BASELINES_DIR}baseline2d_name_desc.csv")

print("Baseline results loaded!")


Baseline results loaded!


In [11]:
# Compare all methods
print("=" * 70)
print("COMPARISON: ALL METHODS")
print("=" * 70)

comparison_results = []

# Baseline 1: Embedding Top 3
b1_found = (baseline1_df['gold_rank'] > 0).sum()
b1_total = len(baseline1_df)
b1_mean = baseline1_df[baseline1_df['gold_rank'] > 0]['gold_rank'].mean()
print(f"\n1. Embedding Top 3:")
print(f"   Gold found: {b1_found}/{b1_total} ({100*b1_found/b1_total:.1f}%)")
print(f"   Mean rank (when found): {b1_mean:.2f}")
comparison_results.append({
    'method': 'Embedding Top 3',
    'found': b1_found, 'total': b1_total,
    'hit_rate': 100*b1_found/b1_total, 'mean_rank': b1_mean
})

# Baseline 2b: LLM + Name Only
b2b_found = (baseline2b_df['gold_rank'] > 0).sum()
b2b_total = len(baseline2b_df)
b2b_mean = baseline2b_df[baseline2b_df['gold_rank'] > 0]['gold_rank'].mean()
print(f"\n2. LLM Selection (Name Only):")
print(f"   Gold found: {b2b_found}/{b2b_total} ({100*b2b_found/b2b_total:.1f}%)")
print(f"   Mean rank (when found): {b2b_mean:.2f}")
comparison_results.append({
    'method': 'LLM: Name Only',
    'found': b2b_found, 'total': b2b_total,
    'hit_rate': 100*b2b_found/b2b_total, 'mean_rank': b2b_mean
})

# Baseline 2c: LLM + Name + Properties + Description
b2c_found = (baseline2c_df['gold_rank'] > 0).sum()
b2c_total = len(baseline2c_df)
b2c_mean = baseline2c_df[baseline2c_df['gold_rank'] > 0]['gold_rank'].mean()
print(f"\n3. LLM Selection (Name + Props + Desc):")
print(f"   Gold found: {b2c_found}/{b2c_total} ({100*b2c_found/b2c_total:.1f}%)")
print(f"   Mean rank (when found): {b2c_mean:.2f}")
comparison_results.append({
    'method': 'LLM: Name + Props + Desc',
    'found': b2c_found, 'total': b2c_total,
    'hit_rate': 100*b2c_found/b2c_total, 'mean_rank': b2c_mean
})

# Baseline 2d: LLM + Name + Description
b2d_found = (baseline2d_df['gold_rank'] > 0).sum()
b2d_total = len(baseline2d_df)
b2d_mean = baseline2d_df[baseline2d_df['gold_rank'] > 0]['gold_rank'].mean()
print(f"\n4. LLM Selection (Name + Desc):")
print(f"   Gold found: {b2d_found}/{b2d_total} ({100*b2d_found/b2d_total:.1f}%)")
print(f"   Mean rank (when found): {b2d_mean:.2f}")
comparison_results.append({
    'method': 'LLM: Name + Desc',
    'found': b2d_found, 'total': b2d_total,
    'hit_rate': 100*b2d_found/b2d_total, 'mean_rank': b2d_mean
})

# Our Pipeline: RAG + LLM Source Mapping (using only successful cases)
print(f"\n5. RAG + LLM Source Mapping Pipeline (OURS):")
print(f"   Gold found: {found}/{successful_count} ({hit_rate:.1f}%)")
if error_count > 0:
    print(f"   ⚠️  {error_count} errors excluded from metrics")
print(f"   Mean rank (when found): {mean_rank:.2f}")
comparison_results.append({
    'method': 'RAG + LLM Source Mapping (OURS)',
    'found': found, 'total': successful_count,  # Use successful_count, not total
    'hit_rate': hit_rate, 'mean_rank': mean_rank,
    'errors': error_count
})

print("\n" + "=" * 70)

# Create comparison DataFrame
comparison_df = pd.DataFrame(comparison_results)
comparison_df = comparison_df.sort_values('hit_rate', ascending=False)
print("\nRanked by Hit Rate:")
print(comparison_df.to_string(index=False))

# Save comparison
comparison_df.to_csv('pipeline_vs_baselines_comparison.csv', index=False)
print("\nComparison saved to pipeline_vs_baselines_comparison.csv")


COMPARISON: ALL METHODS

1. Embedding Top 3:
   Gold found: 160/400 (40.0%)
   Mean rank (when found): 1.75

2. LLM Selection (Name Only):
   Gold found: 129/400 (32.2%)
   Mean rank (when found): 1.65

3. LLM Selection (Name + Props + Desc):
   Gold found: 134/400 (33.5%)
   Mean rank (when found): 1.55

4. LLM Selection (Name + Desc):
   Gold found: 130/400 (32.5%)
   Mean rank (when found): 1.62

5. RAG + LLM Source Mapping Pipeline (OURS):
   Gold found: 131/389 (33.7%)
   ⚠️  11 errors excluded from metrics
   Mean rank (when found): 1.63


Ranked by Hit Rate:
                         method  found  total  hit_rate  mean_rank  errors
                Embedding Top 3    160    400 40.000000   1.750000     NaN
RAG + LLM Source Mapping (OURS)    131    389 33.676093   1.633588    11.0
       LLM: Name + Props + Desc    134    400 33.500000   1.552239     NaN
               LLM: Name + Desc    130    400 32.500000   1.615385     NaN
                 LLM: Name Only    129    400 32.2500

In [12]:
# Show some example results with reasoning
print("\n" + "="*70)
print("SAMPLE RESULTS (with reasoning)")
print("="*70)

for i in range(min(3, len(results))):
    r = results[i]
    print(f"\n{'─'*70}")
    print(f"[{i+1}] Target: {r['target']}")
    print(f"    Gold Source: {r['gold_source']}")
    print(f"    Status: {r['status']}")
    print(f"    Gold Rank: {r['gold_rank']}")
    
    if r['selected_sources'] != ERROR_MARKER:
        print(f"\n    Selected Sources: {r['selected_sources']}")
        
        if r['selected_mappings'] and len(r['selected_mappings']) > 0:
            print(f"\n    Top Selection Mappings:")
            for j, (src, mapping) in enumerate(zip(r['selected_sources'], r['selected_mappings'])):
                if isinstance(mapping, dict):
                    print(f"      {j+1}. {src}: {mapping}")
        
        if r.get('reranker_reasoning'):
            reasoning = r['reranker_reasoning']
            print(f"\n    Reranker Reasoning:")
            # Truncate long reasoning
            if len(reasoning) > 300:
                print(f"      {reasoning[:300]}...")
            else:
                print(f"      {reasoning}")
    
    # Show one mapping reasoning sample
    if r.get('mapping_reasonings') and r['mapping_reasonings'] != ERROR_MARKER:
        valid_reasonings = [mr for mr in r['mapping_reasonings'] if not str(mr).startswith('ERROR')]
        if valid_reasonings:
            print(f"\n    Sample Mapping Reasoning (1st successful):")
            sample = valid_reasonings[0]
            if len(sample) > 200:
                print(f"      {sample[:200]}...")
            else:
                print(f"      {sample}")

print(f"\n{'─'*70}")



SAMPLE RESULTS (with reasoning)

──────────────────────────────────────────────────────────────────────
[1] Target: biological clock
    Gold Source: clock
    Status: success
    Gold Rank: 1

    Selected Sources: ['clock', 'electronic scale automatically adjusts', 'Human Body']

    Top Selection Mappings:
      1. clock: {'changes': 'movement', 'state': 'time', 'adjust': 'setting'}
      2. electronic scale automatically adjusts: {'changes': 'calibration', 'state': 'measurement', 'adjust': 'auto-adjustment'}
      3. Human Body: {'changes': 'metabolism', 'state': 'homeostasis', 'adjust': 'regulation'}

    Reranker Reasoning:
      The target concept "biological clock" has properties related to changes, state, and adjustment. The best analogous source concepts should have properties that map well to these, showing a strong structural or functional correspondence. The selected candidates should also be familiar and help explain...

    Sample Mapping Reasoning (1st successful):
   